# examples-seen-step-axis — worked example 1: Cumulative examples_seen Counter Inside a Training Loop

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `examples-seen-step-axis`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When logging training metrics to a tracking tool like Weights & Biases, using the step number as the x-axis makes it impossible to fairly compare runs with different batch sizes. A run with batch size 64 takes half as many steps as one with batch size 32 to process the same data. Using `examples_seen = step * batch_size` as the x-axis ensures both runs align when they have processed the same amount of training data, regardless of batch size.

## Worked solution

We simulate a short training run and convert the step-indexed logs to examples_seen.

**Step 1:** We have 6 steps with batch_size=16. `examples_seen` at each step is simply `step * 16`.
- Step 1 → 16 examples, Step 2 → 32, ..., Step 6 → 96.

**Step 2:** The same 96 total examples processed with batch_size=32 takes only 3 steps.
- Step 1 → 32, Step 2 → 64, Step 3 → 96.

**Step 3:** Both runs reach `examples_seen = 96` at their last log point. A loss-vs-examples_seen plot would correctly show both runs' final loss at the same x-coordinate, even though one has 6 steps and the other has 3.

This comparability property is why ARENA uses examples_seen rather than the raw step counter as the wandb x-axis.

In [ ]:
import torch as t

t.manual_seed(1)

batch_size_a = 16
batch_size_b = 32

# Simulated loss histories (step, loss)
history_a = [(s, 1.0 / s) for s in range(1, 7)]   # 6 steps, batch=16
history_b = [(s, 1.2 / s) for s in range(1, 4)]   # 3 steps, batch=32

def to_examples_seen(history, batch_size):
    return [(step * batch_size, loss) for step, loss in history]

log_a = to_examples_seen(history_a, batch_size_a)
log_b = to_examples_seen(history_b, batch_size_b)

print("Run A (bs=16) examples_seen x-values:", [ex for ex, _ in log_a])
print("Run B (bs=32) examples_seen x-values:", [ex for ex, _ in log_b])

# Both should reach 96 total examples
assert log_a[-1][0] == 96
assert log_b[-1][0] == 96
print("Both runs end at examples_seen = 96 — the x-axes are comparable.")

# Verify values explicitly
for (ex, _), step in zip(log_a, range(1, 7)):
    assert ex == step * batch_size_a
print("examples_seen values verified for run A.")